# Notebook 7: Cross-Person Generalization Experiment

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [2]:
base = r"C:\Users\M Mayavan\OneDrive\Desktop\HAR_Project\UCI HAR Dataset"
X_train_all = np.loadtxt(base + r"\train\X_train.txt")
y_train_all = np.loadtxt(base + r"\train\y_train.txt") - 1
subject_train = np.loadtxt(base + r"\train\subject_train.txt")
X_test = np.loadtxt(base + r"\test\X_test.txt")
y_test = np.loadtxt(base + r"\test\y_test.txt") - 1
print("Training subjects:", np.unique(subject_train))
print("Total training samples:", X_train_all.shape[0])

Training subjects: [ 1.  3.  5.  6.  7.  8. 11. 14. 15. 16. 17. 19. 21. 22. 23. 25. 26. 27.
 28. 29. 30.]
Total training samples: 7352


In [3]:
mask_train = (subject_train <= 27)
mask_cross = (subject_train >= 28)
X_train = X_train_all[mask_train]
y_train = y_train_all[mask_train]
X_cross = X_train_all[mask_cross]
y_cross = y_train_all[mask_cross]
print("Training samples (subjects 1-27):", X_train.shape[0])
print("Cross-person samples (subjects 28-30):", X_cross.shape[0])
print("Official test samples:", X_test.shape[0])

Training samples (subjects 1-27): 6243
Cross-person samples (subjects 28-30): 1109
Official test samples: 2947


In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_cross_s = scaler.transform(X_cross)
X_test_s = scaler.transform(X_test)
X_train_r = X_train_s.reshape(-1, 33, 17)
X_cross_r = X_cross_s.reshape(-1, 33, 17)
X_test_r = X_test_s.reshape(-1, 33, 17)
X_train_t = torch.tensor(X_train_r, dtype=torch.float32)
X_cross_t = torch.tensor(X_cross_r, dtype=torch.float32)
X_test_t = torch.tensor(X_test_r, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_cross_t = torch.tensor(y_cross, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [5]:
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
cross_loader = DataLoader(TensorDataset(X_cross_t, y_cross_t), batch_size=64)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64)

In [6]:
class LSTM(nn.Module):
    def __init__(self):
        super(LSTM, self).__init__()
        self.lstm = nn.LSTM(17, 128, num_layers=2, batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(128, 6)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.drop(out)
        out = self.fc(out)
        return out

In [7]:
model = LSTM()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss)

1 115.43280398845673
2 72.92060476541519
3 55.94482943415642
4 42.130816489458084
5 33.70944467186928
6 26.797558158636093
7 22.048158563673496
8 18.22149208188057
9 16.294120650738478
10 14.013226974755526
11 13.394213169813156
12 12.722951978445053
13 11.300709608010948
14 11.800914097577333
15 9.23506904952228
16 9.187910029664636
17 8.643733809702098
18 8.704897550866008
19 7.800299287773669
20 6.378455963917077
21 7.105407170020044
22 6.648894145153463
23 6.01451751543209
24 5.734284066129476
25 4.876927434001118
26 5.198664207244292
27 4.736867762403563
28 5.3611876158975065
29 4.375418781070039
30 6.0621829754672945
31 4.266114000696689
32 3.6285182707943022
33 3.3665560795925558
34 3.173331225523725
35 2.520565129816532
36 3.1773670280817896
37 2.409506499650888
38 2.6827025045640767
39 2.2073666447540745
40 2.1192289070459083
41 2.143090161087457
42 2.108832458499819
43 2.3429732795339078
44 1.2444565357000101
45 1.5840856314753182
46 1.778381530661136
47 2.2529078797088005
48

In [8]:
model.eval()
preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds.append(torch.argmax(output, dim=1))
preds = torch.cat(preds)
print("Standard Test (official test set):")
print(classification_report(y_test_t, preds))

Standard Test (official test set):
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       496
           1       0.90      0.88      0.89       471
           2       0.89      0.90      0.90       420
           3       0.90      0.88      0.89       491
           4       0.88      0.90      0.89       532
           5       1.00      0.99      1.00       537

    accuracy                           0.91      2947
   macro avg       0.91      0.91      0.91      2947
weighted avg       0.92      0.91      0.91      2947



In [9]:
model.eval()
preds = []
with torch.no_grad():
    for X_batch, y_batch in cross_loader:
        output = model(X_batch)
        preds.append(torch.argmax(output, dim=1))
preds = torch.cat(preds)
print("Cross-Person Test (subjects 28-30 held out from training):")
print(classification_report(y_cross_t, preds))

Cross-Person Test (subjects 28-30 held out from training):
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       172
           1       0.99      0.99      0.99       165
           2       1.00      0.98      0.99       156
           3       0.89      0.78      0.83       194
           4       0.81      0.91      0.86       203
           5       1.00      1.00      1.00       219

    accuracy                           0.94      1109
   macro avg       0.95      0.94      0.94      1109
weighted avg       0.94      0.94      0.94      1109

